# Entities: the domain-general vocabulary

| marketing (parent) | axiom |
|---|---|
| channel | `Treatment` |
| spend / impressions | `Dose` |
| geo / DMA | `Unit` |
| KPI / sales | `Outcome` |
| control variable | `Covariate` |

Every entity carries a `Dimension` and optionally a unit of measure (a plain string that the
`UnitSystem` knows). `Dose` also carries a `numeraire`. `Population`, `TimeWindow`, and
`Intervention` are the scope-and-intervention vocabulary the estimand facets are built on.

*(Notebook 03 in this series — the expression tree and its interpreters — lands in Phase 1b.)*

In [ ]:
import warnings

from axiom.core import (
    Covariate,
    D,
    Dose,
    Entity,
    EntityName,
    Intervention,
    Outcome,
    Population,
    TimeWindow,
    Treatment,
    UndimensionedWarning,
    Unit,
    dimension_of,
    dimensionless,
)

from axiom.display import enable

enable();  # every axiom result renders itself from here on

In [ ]:
fertilizer = Treatment(name="fertilizer", dimension=D.currency, unit="USD", description="applied nitrogen, costed")
dose = Dose(name="fertilizer_dose", dimension=D.currency, unit="USD", numeraire="USD")
plot = Unit(name="plot", dimension=D.entity, kind="cluster")
yield_total = Outcome(name="yield_total", dimension=D.outcome, unit="kg", aggregation="sum")
rainfall = Covariate(name="rainfall", dimension=dimensionless(), description="standardized")

for e in (fertilizer, dose, plot, yield_total, rainfall):
    print(f"{type(e).__name__:10s} {e.name:16s} dim={e.dim!s:6s} unit={e.unit}")

## `Entity` is a protocol, not a base class

The five entity specs are independent classes (composition over inheritance: they share the
`EntityName` field type and one validator function, not a parent). `Entity` is the protocol
any of them satisfies, so a function can accept "any entity" without a hierarchy.

In [ ]:
def describe(e: Entity) -> str:
    return f"{e.name} [{dimension_of(e)}]"

print([describe(e) for e in (fertilizer, plot, rainfall)])
print(isinstance(dose, Entity), Treatment.__mro__[1].__name__)

name: EntityName = "tv-ads.v2"
print(name)

## Undimensioned user entities warn once (D6)

User code may omit the dimension; the entity is then dimensionless and a warning says so.
Anything *shipped* in `src/axiom` must be dimensioned — gate 10 checks that separately.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    loose = Covariate(name="something")
print(loose.dim, "|", [w.category.__name__ for w in caught], "|", issubclass(caught[0].category, UndimensionedWarning))

## Population, window, intervention

These three are facets of an estimand (Phase 1b). Note what each pins down that a variable
name would not: strata weights on the population, the time *basis* on the window, the
treatment *version* on the intervention (review B2 — SUTVA-1).

In [ ]:
north = Population(name="north", strata={"soil": {"clay": 0.3, "loam": 0.7}})
season = TimeWindow(start=0, stop=13, basis="cumulative")
iv = Intervention(doses={"fertilizer": 120.0}, mode="set", version="granular-v2", window=season)

print(north.strata, "|", season.length, season.basis)
print(iv.treatments, iv.mode, iv.version)
print("same dose, different version, different intervention:",
      iv != iv.model_copy(update={"version": "liquid-v1"}))

In [ ]:
try:
    Population(name="bad", strata={"soil": {"clay": 0.5, "loam": 0.6}})
except ValueError as e:
    print("refused:", e)
try:
    TimeWindow(start=5, stop=5)
except ValueError as e:
    print("refused:", e)